## 1) Cấu hình & Kết nối MySQL

In [19]:
# === CONFIG ===
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "123456"

DB_USER  = "bookweb_user"
DB_BOOK  = "bookweb_book"
DB_ORDER = "bookweb_order"

# In nhanh
PRINT_MAX_ITEMS = 10
PRINT_MAX_USERS = 5 # Giảm xuống 5 user để tiện theo dõi các ma trận trung gian
TOP_K = 8

# === Thư viện ===
import numpy as np, pandas as pd
from sqlalchemy import create_engine, text
from collections import defaultdict

def get_engine():
    dsn = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}"
    return create_engine(dsn, pool_pre_ping=True, pool_recycle=1800)

def fetch_all(sql: str, params: dict | None = None):
    with engine.connect() as conn:
        res = conn.execute(text(sql), params or {})
        cols = res.keys()
        return [dict(zip(cols, row)) for row in res.fetchall()]

engine = get_engine()
print("Kết nối CSDL đã sẵn sàng.")


Kết nối CSDL đã sẵn sàng.


## 2) Nạp dữ liệu

In [20]:
authors = fetch_all(f"SELECT id, name FROM {DB_BOOK}.authors")
categories = fetch_all(f"SELECT id, name FROM {DB_BOOK}.categories")
publishers = fetch_all(f"SELECT id, name FROM {DB_BOOK}.publishers")

author2idx = {r['id']: i for i, r in enumerate(authors)}
cate2idx   = {r['id']: i for i, r in enumerate(categories)}
pub2idx    = {r['id']: i for i, r in enumerate(publishers)}

idx2author = {i: r['name'] for i, r in enumerate(authors)}
idx2cate   = {i: r['name'] for i, r in enumerate(categories)}
idx2pub    = {i: r['name'] for i, r in enumerate(publishers)}

ab = fetch_all(f"SELECT book_id, author_id    FROM {DB_BOOK}.author_book")
bc = fetch_all(f"SELECT book_id, category_id FROM {DB_BOOK}.book_category")
bp = fetch_all(f"SELECT id AS book_id, publisher_id, title FROM {DB_BOOK}.books")

book_authors = defaultdict(list)
for r in ab:
    if r['author_id'] in author2idx:
        book_authors[r['book_id']].append(author2idx[r['author_id']])

book_cates = defaultdict(list)
for r in bc:
    if r['category_id'] in cate2idx:
        book_cates[r['book_id']].append(cate2idx[r['category_id']])

book_pubs = {}
book_meta = {}
book_ids = []
for r in bp:
    bid = r['book_id']
    book_ids.append(bid)
    if r['publisher_id'] in pub2idx:
        book_pubs[bid] = pub2idx[r['publisher_id']]
    book_meta[bid] = {'title': r['title']}

na, nc, npub = len(author2idx), len(cate2idx), len(pub2idx)
print("Kích thước danh mục:", "Authors=", na, "Categories=", nc, "Publishers=", npub, "| Books=", len(book_ids))


Kích thước danh mục: Authors= 18 Categories= 15 Publishers= 10 | Books= 30


## 3) Item profile
Ma trận item nhị phân cho từng khối (A_items, C_items, P_items)

In [21]:
def build_item_matrices():
    A = np.zeros((len(book_ids), na), dtype=float)
    C = np.zeros((len(book_ids), nc), dtype=float)
    P = np.zeros((len(book_ids), npub), dtype=float)
    for row_idx, bid in enumerate(book_ids):
        for ai in book_authors.get(bid, []): A[row_idx, ai] = 1.0
        for ci in book_cates.get(bid, []):    C[row_idx, ci] = 1.0
        pj = book_pubs.get(bid, None)
        if pj is not None: P[row_idx, pj] = 1.0
    return A, C, P

A_items, C_items, P_items = build_item_matrices()

A_cols = [f"A:{idx2author[i]}" for i in range(na)]
C_cols = [f"C:{idx2cate[i]}"    for i in range(nc)]
P_cols = [f"P:{idx2pub[i]}"     for i in range(npub)]

# FIX: Định nghĩa hàm hiển thị DataFrame item
def df_items_block(M, cols):
    return pd.DataFrame(
        M[:PRINT_MAX_ITEMS, :],
        index=[f"{book_ids[i]}:{book_meta[book_ids[i]]['title']}" for i in range(min(PRINT_MAX_ITEMS, len(book_ids)))],
        columns=cols
    )

print("A_items:", A_items.shape, "| C_items:", C_items.shape, "| P_items:", P_items.shape)
df_items_block(A_items, A_cols).loc[:, (A_items[:PRINT_MAX_ITEMS]!=0).any(axis=0)]


A_items: (30, 18) | C_items: (30, 15) | P_items: (30, 10)


,A:Tô Hoài,A:Paulo CoeHo,A:Dale Carnegie,A:Rosie Nguyễn,A:Patrick Modiano,A:Hyun-wook park,A:Xuân Quỳnh
1:Dế Mèn phưu lưu kí,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2:Hoàng Tử Bé,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3:Đắc nhân tâm,0.0,0.0,1.0,0.0,0.0,0.0,0.0
10:Công chúa ngủ trong rừng,1.0,0.0,0.0,0.0,0.0,0.0,0.0
14:Tuổi trẻ đáng giá bao nhiêu?,0.0,0.0,0.0,1.0,0.0,0.0,0.0
15:Tuần trăng mật,0.0,0.0,0.0,0.0,1.0,0.0,0.0
16:Giã từ thơ ngây,0.0,0.0,0.0,0.0,0.0,1.0,0.0
17:Hồi ức là cuộn băng tua ngược,0.0,0.0,0.0,0.0,0.0,1.0,0.0
18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,0.0,0.0,0.0,0.0,0.0,0.0,1.0
19:KHÔNG BAO GIỜ LÀ CUỐI,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [22]:
# FIX: Gọi lại hàm
df_items_block(C_items, C_cols).loc[:, (C_items[:PRINT_MAX_ITEMS]!=0).any(axis=0)]


,C:Tiểu thuyết,C:Hài hước,C:Kỹ năng,C:Thiếu nhi,C:Lãng mạn,C:Thơ - Kịch
1:Dế Mèn phưu lưu kí,1.0,0.0,0.0,1.0,0.0,0.0
2:Hoàng Tử Bé,1.0,1.0,0.0,0.0,0.0,0.0
3:Đắc nhân tâm,0.0,0.0,1.0,0.0,0.0,0.0
10:Công chúa ngủ trong rừng,0.0,0.0,0.0,1.0,0.0,0.0
14:Tuổi trẻ đáng giá bao nhiêu?,0.0,0.0,1.0,0.0,0.0,0.0
15:Tuần trăng mật,1.0,1.0,0.0,0.0,0.0,0.0
16:Giã từ thơ ngây,1.0,0.0,0.0,0.0,1.0,0.0
17:Hồi ức là cuộn băng tua ngược,1.0,1.0,0.0,0.0,1.0,0.0
18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,0.0,0.0,0.0,0.0,0.0,1.0
19:KHÔNG BAO GIỜ LÀ CUỐI,0.0,0.0,0.0,0.0,0.0,1.0


In [23]:
# FIX: Gọi lại hàm
df_items_block(P_items, P_cols).loc[:, (P_items[:PRINT_MAX_ITEMS]!=0).any(axis=0)]


,P:Kim Đồng,P:Nhã Nam,P:NXB Trẻ,P:Hội nhà văn
1:Dế Mèn phưu lưu kí,1.0,0.0,0.0,0.0
2:Hoàng Tử Bé,1.0,0.0,0.0,0.0
3:Đắc nhân tâm,0.0,0.0,1.0,0.0
10:Công chúa ngủ trong rừng,1.0,0.0,0.0,0.0
14:Tuổi trẻ đáng giá bao nhiêu?,0.0,0.0,1.0,0.0
15:Tuần trăng mật,0.0,0.0,0.0,1.0
16:Giã từ thơ ngây,0.0,1.0,0.0,0.0
17:Hồi ức là cuộn băng tua ngược,0.0,0.0,1.0,0.0
18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,0.0,0.0,0.0,1.0
19:KHÔNG BAO GIỜ LÀ CUỐI,0.0,0.0,0.0,1.0


## 4) Ma trận **W** (user–item) — review & favorite

In [24]:
# Tập user từ reviews ∪ favorites
uids_reviews = fetch_all(f"SELECT DISTINCT buyer_id AS uid FROM {DB_ORDER}.reviews")
uids_favs    = fetch_all(f"SELECT DISTINCT buyer_id AS uid FROM {DB_ORDER}.favorites")
user_ids = sorted(list({r['uid'] for r in (uids_reviews + uids_favs)}))

uid2row = {uid: i for i, uid in enumerate(user_ids)}
bid2col = {bid: j for j, bid in enumerate(book_ids)}

W = np.zeros((len(user_ids), len(book_ids)), dtype=float)

# Review mới nhất cho mỗi (user, book)
latest_reviews = fetch_all(f"""
    SELECT buyer_id AS uid, book_id, stars AS rating
    FROM (
        SELECT buyer_id, book_id, stars,
               ROW_NUMBER() OVER (PARTITION BY buyer_id, book_id ORDER BY created_at DESC, id DESC) rn
        FROM {DB_ORDER}.reviews
    ) t
    WHERE rn = 1
""")
for r in latest_reviews:
    uid, bid, rating = r['uid'], r['book_id'], float(r['rating'])
    if uid in uid2row and bid in bid2col:
        W[uid2row[uid], bid2col[bid]] = max(W[uid2row[uid], bid2col[bid]], rating)

# Favorites = 5 điểm
favs = fetch_all(f"SELECT buyer_id AS uid, book_id FROM {DB_ORDER}.favorites")
for r in favs:
    uid, bid = r['uid'], r['book_id']
    if uid in uid2row and bid in bid2col:
        W[uid2row[uid], bid2col[bid]] = max(W[uid2row[uid], bid2col[bid]], 5.0)

print("W shape:", W.shape, "| #users:", len(user_ids), "| #items:", len(book_ids))

# In vài hàng đầu (ẩn cột toàn 0)
dfW = pd.DataFrame(
    W[:PRINT_MAX_USERS, :PRINT_MAX_ITEMS],
    index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))],
    columns=[f"{book_ids[j]}:{book_meta[book_ids[j]]['title']}" for j in range(min(PRINT_MAX_ITEMS, len(book_ids)))]
)
dfW.loc[:, (dfW.abs() > 0).any(axis=0)]


W shape: (10, 30) | #users: 10 | #items: 30


,1:Dế Mèn phưu lưu kí,3:Đắc nhân tâm,14:Tuổi trẻ đáng giá bao nhiêu?
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,0.0,0.0
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0,0.0
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,0.0,0.0,0.0
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,0.0,3.0
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.0,5.0,0.0


## 5) Vector user theo từng khối (Raw User Profiles: A_user_raw, C_user_raw, P_user_raw)

Tính **Tổng điểm** sở thích cho từng thuộc tính: $\mathbf{U}_{\text{raw}} = \mathbf{W} \times \mathbf{U}_{\text{items}}$


In [25]:
# Tính Vector Sở thích Thô (Raw User Profiles - Tổng điểm)
A_user_raw = W @ A_items
C_user_raw = W @ C_items
P_user_raw = W @ P_items

print("A_user_raw:", A_user_raw.shape, "| C_user_raw:", C_user_raw.shape, "| P_user_raw:", P_user_raw.shape)

# FIX: Định nghĩa hàm hiển thị DataFrame raw
def df_user_block_raw(M, cols, block_name):
    df = pd.DataFrame(
        M[:PRINT_MAX_USERS, :],
        index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))],
        columns=cols
    )
    print(f"\n--- VÍ DỤ: {block_name} (Tổng điểm Thô) của 5 User đầu tiên ---")
    return df.loc[:, (df.abs() > 0).any(axis=0)]

df_user_block_raw(A_user_raw, A_cols, "Vector A_user_raw - Tác giả")


A_user_raw: (10, 18) | C_user_raw: (10, 15) | P_user_raw: (10, 10)

--- VÍ DỤ: Vector A_user_raw - Tác giả (Tổng điểm Thô) của 5 User đầu tiên ---


,A:Tô Hoài,A:Dale Carnegie,A:Rosie Nguyễn,A:J. R. R. Tolkien,A:Laura Cowan,A:Stephen Hawking,A: Phương Hoài Nga,A: Alexandre Dumas,A:Trần Lỗi,A:Benjamin Graham,A:Raymond Chandler
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0,0.0,0.0,5.0,0.0,5.0,0.0,0.0,4.0,0.0
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,0.0,3.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,3.0,0.0


In [26]:
df_user_block_raw(C_user_raw, C_cols, "Vector C_user_raw - Thể loại")



--- VÍ DỤ: Vector C_user_raw - Thể loại (Tổng điểm Thô) của 5 User đầu tiên ---


,C:Tiểu thuyết,C:Hài hước,C:Kỹ năng,C:Thiếu nhi,C:Giáo trình,C:Khoa học,C:Văn học,C:Phiêu lưu,C:Kiến thức,C:Truyện tranh,C:Tài chính - Kinh doanh,C:Trinh thám
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,5.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,5.0,0.0,0.0
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0,9.0,5.0,5.0,5.0,0.0,0.0,14.0,0.0,4.0,0.0
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,10.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,0.0,0.0,0.0,5.0
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,4.0,3.0,4.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.0,0.0,5.0,0.0,0.0,5.0,0.0,0.0,5.0,0.0,3.0,0.0


In [27]:
df_user_block_raw(P_user_raw, P_cols, "Vector P_user_raw - NXB")



--- VÍ DỤ: Vector P_user_raw - NXB (Tổng điểm Thô) của 5 User đầu tiên ---


,P:Kim Đồng,P:NXB Trẻ,P:Hội nhà văn,P:Lao Động,P:Văn học,P:Dân Trí,P:Thế Giới,P:Phụ Nữ
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0,0.0,4.0,0.0,5.0,0.0,5.0
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,3.0,4.0,0.0,0.0,0.0,0.0,0.0
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.0,5.0,0.0,3.0,0.0,0.0,5.0,0.0


## 5.1) User profile


In [28]:
def normalize_user_profile(U_raw: np.ndarray) -> np.ndarray:
    row_sums = U_raw.sum(axis=1, keepdims=True)
    U_norm = U_raw.copy()
    
    # Xác định các hàng có tổng khác 0 để thực hiện chuẩn hóa
    non_zero_sum_mask = (row_sums.flatten() != 0)
    
    # Chia chỉ các hàng có tổng khác 0
    U_norm[non_zero_sum_mask] = U_raw[non_zero_sum_mask] / row_sums[non_zero_sum_mask]
    
    return U_norm

# Áp dụng chuẩn hóa và gán lại biến sử dụng trong tính toán Cosine
A_user = normalize_user_profile(A_user_raw)
C_user = normalize_user_profile(C_user_raw)
P_user = normalize_user_profile(P_user_raw)

# Định nghĩa hàm hiển thị DataFrame normalized
def df_user_block_norm(M, cols, block_name):
    df = pd.DataFrame(
        M[:PRINT_MAX_USERS, :],
        index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))],
        columns=cols
    )
    print(f"\n--- VÍ DỤ: {block_name} (Chuẩn hóa) của 5 User đầu tiên ---")
    return df.loc[:, (df.abs() > 0).any(axis=0)]

df_user_block_norm(A_user, A_cols, "Vector A_user - Tác giả")



--- VÍ DỤ: Vector A_user - Tác giả (Chuẩn hóa) của 5 User đầu tiên ---


,A:Tô Hoài,A:Dale Carnegie,A:Rosie Nguyễn,A:J. R. R. Tolkien,A:Laura Cowan,A:Stephen Hawking,A: Phương Hoài Nga,A: Alexandre Dumas,A:Trần Lỗi,A:Benjamin Graham,A:Raymond Chandler
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.0
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.000000,0.000000,0.000000,0.0,0.357143,0.000000,0.357143,0.000000,0.0,0.285714,0.0
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,0.000000,0.000000,0.000000,0.5,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.5
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,0.363636,0.000000,0.272727,0.0,0.000000,0.000000,0.000000,0.363636,0.0,0.000000,0.0
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.000000,0.384615,0.000000,0.0,0.000000,0.384615,0.000000,0.000000,0.0,0.230769,0.0


In [29]:
df_user_block_norm(C_user, C_cols, "Vector C_user - Thể loại")



--- VÍ DỤ: Vector C_user - Thể loại (Chuẩn hóa) của 5 User đầu tiên ---


,C:Tiểu thuyết,C:Hài hước,C:Kỹ năng,C:Thiếu nhi,C:Giáo trình,C:Khoa học,C:Văn học,C:Phiêu lưu,C:Kiến thức,C:Truyện tranh,C:Tài chính - Kinh doanh,C:Trinh thám
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,0.0,0.333333,0.000000,0.333333,0.000000,0.0
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.000000,0.000000,0.214286,0.119048,0.119048,0.119048,0.0,0.000000,0.333333,0.000000,0.095238,0.0
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,0.400000,0.000000,0.000000,0.000000,0.000000,0.000000,0.2,0.200000,0.000000,0.000000,0.000000,0.2
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,0.173913,0.173913,0.130435,0.173913,0.000000,0.000000,0.0,0.173913,0.000000,0.173913,0.000000,0.0
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.000000,0.000000,0.277778,0.000000,0.000000,0.277778,0.0,0.000000,0.277778,0.000000,0.166667,0.0


In [30]:
df_user_block_norm(P_user, P_cols, "Vector P_user - NXB")



--- VÍ DỤ: Vector P_user - NXB (Chuẩn hóa) của 5 User đầu tiên ---


,P:Kim Đồng,P:NXB Trẻ,P:Hội nhà văn,P:Lao Động,P:Văn học,P:Dân Trí,P:Thế Giới,P:Phụ Nữ
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.000000,1.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.000000,0.000000,0.000000,0.285714,0.0,0.357143,0.000000,0.357143
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,0.363636,0.272727,0.363636,0.000000,0.0,0.000000,0.000000,0.000000
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.000000,0.384615,0.000000,0.230769,0.0,0.000000,0.384615,0.000000


## 6) Cosine từng khối + Trung bình cộng

In [31]:
def safe_cos(u_vec: np.ndarray, i_vec: np.ndarray) -> float:
    nu = np.linalg.norm(u_vec); ni = np.linalg.norm(i_vec)
    if nu == 0.0 or ni == 0.0:
        return 0.0
    # Các vector U_user đã được CHUẨN HÓA (theo tổng điểm) ở Mục 5.1
    # Hàm này tính Cosine Similarity trên vector đã được chuẩn hóa đó.
    return float(u_vec.dot(i_vec) / (nu * ni))

def score_user_item(u_idx: int, i_idx: int) -> float:
    sA = safe_cos(A_user[u_idx, :], A_items[i_idx, :])
    sC = safe_cos(C_user[u_idx, :], C_items[i_idx, :])
    sP = safe_cos(P_user[u_idx, :], P_items[i_idx, :])
    return (sA + sC + sP) / 3.0

def build_scores_matrix():
    m, n = W.shape
    S = np.zeros((m, n), dtype=float)
    for u in range(m):
        for i in range(n):
            S[u, i] = score_user_item(u, i)
    return S

S = build_scores_matrix()
print("S shape:", S.shape)

dfS = pd.DataFrame(
    S[:PRINT_MAX_USERS, :PRINT_MAX_ITEMS],
    index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))],
    columns=[f"{book_ids[j]}:{book_meta[book_ids[j]]['title']}" for j in range(min(PRINT_MAX_ITEMS, len(book_ids)))]
)
dfS.loc[:, (dfS.abs() > 0).any(axis=0)]


S shape: (10, 30)


,1:Dế Mèn phưu lưu kí,2:Hoàng Tử Bé,3:Đắc nhân tâm,10:Công chúa ngủ trong rừng,14:Tuổi trẻ đáng giá bao nhiêu?,15:Tuần trăng mật,16:Giã từ thơ ngây,17:Hồi ức là cuộn băng tua ngược,18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,19:KHÔNG BAO GIỜ LÀ CUỐI
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.000000,0.136083,0.333333,0.000000,0.333333,0.136083,0.000000,0.444444,0.000000,0.000000
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.061434,0.000000,0.156386,0.086881,0.156386,0.000000,0.000000,0.000000,0.000000,0.000000
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,0.178174,0.178174,0.000000,0.000000,0.000000,0.178174,0.178174,0.145479,0.000000,0.000000
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,0.616338,0.408107,0.262174,0.557796,0.418347,0.408107,0.099938,0.319371,0.208232,0.208232
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.000000,0.000000,0.615811,0.000000,0.398830,0.000000,0.000000,0.216982,0.000000,0.000000


## 7) Recommend Top-K (loại item đã tương tác)

In [32]:
def recommend_for_user(u_idx: int, top_k=TOP_K, exclude_interacted=True):
    m, n = W.shape
    inter = set(np.where(W[u_idx, :] > 0)[0]) if exclude_interacted else set()
    rows = []
    for j in range(n):
        if j in inter:
            continue
        rows.append((j, float(S[u_idx, j])))
    rows.sort(key=lambda x: x[1], reverse=True)
    rows = rows[:top_k]
    out = []
    for j, sc in rows:
        bid = book_ids[j]
        out.append({"book_id": bid, "title": book_meta[bid]["title"], "score": sc})
    return out

if len(user_ids) > 0:
    u0 = 1
    print(f"User u{u0}: {user_ids[u0]}")
    recs_df = pd.DataFrame(recommend_for_user(u0, top_k=TOP_K, exclude_interacted=True))
    recs_df


User u1: 44475e62-c856-46fa-b705-fb8ee2fbf444


## 8) Demo một user

In [35]:
def list_interactions(u_idx: int):
    js = np.where(W[u_idx, :] > 0)[0]
    rows = []
    for j in js:
        bid = book_ids[j]
        rows.append({"book_id": bid, "title": book_meta[bid]["title"], "rating": float(W[u_idx, j])})
    rows.sort(key=lambda x: -x["rating"])
    return rows

def top_features_block(vec, names, k=10):
    nz = [(names[i], float(vec[i])) for i in np.where(vec != 0)[0]]
    nz.sort(key=lambda x: abs(x[1]), reverse=True)
    return nz[:k]

def explain_user(u_idx: int, k_feat=10, sample_item_idx=None):
    print(f"=== USER u{u_idx}: {user_ids[u_idx]} ===")
    rows = list_interactions(u_idx)
    print("\nCác sách đã tương tác:")
    for r in rows:
        print(f"  - {r['rating']:>4.1f} × [{r['book_id']}] {r['title']}")

    # SỬ DỤNG A_USER ĐÃ CHUẨN HÓA (Mục 5.1)
    print("\nTỷ lệ sở thích đối với Tổng điểm tác giả")
    for name, val in top_features_block(A_user[u_idx,:], A_cols, k=k_feat):
        print(f"  - {name:40s} {val: .3f}")

    print("\nTỷ lệ sở thích đối với Tổng điểm thể loại")
    for name, val in top_features_block(C_user[u_idx,:], C_cols, k=k_feat):
        print(f"  - {name:40s} {val: .3f}")

    print("\nTỷ lệ sở thích đối với Tổng điểm NXB")
    for name, val in top_features_block(P_user[u_idx,:], P_cols, k=min(k_feat,5)):
        print(f"  - {name:40s} {val: .3f}")

    if sample_item_idx is not None:
        sA = safe_cos(A_user[u_idx,:], A_items[sample_item_idx,:])
        sC = safe_cos(C_user[u_idx,:], C_items[sample_item_idx,:])
        sP = safe_cos(P_user[u_idx,:], P_items[sample_item_idx,:])
        total = (sA + sC + sP) / 3.0
        bid = book_ids[sample_item_idx]
        print(f"\nĐiểm của cuốn sách gần nhất [{bid}] {book_meta[bid]['title']}:")
        print(f"  cos_A={sA:.4f}  |  cos_C={sC:.4f}  |  cos_P={sP:.4f}")
        print(f"  => AVERAGE = (cos_A + cos_C + cos_P) / 3 = {total:.6f}")

# Demo: giải thích user 2 và phân rã điểm của gợi ý đầu tiên
if len(user_ids) > 0:
    u0 = 2
    recs = recommend_for_user(u0, top_k=TOP_K, exclude_interacted=True)
    sample_idx = None
    if len(recs) > 0:
        b2j = {book_ids[j]: j for j in range(len(book_ids))}
        sample_idx = b2j[recs[0]["book_id"]]
    explain_user(u0, k_feat=10, sample_item_idx=sample_idx)


=== USER u2: 55575e62-c856-46fa-b705-fb8ee2fbf555 ===

Các sách đã tương tác:
  -  5.0 × [20] CHÚA TỂ NHỮNG CHIẾC NHẪN TẬP 1 - ĐOÀN HỘ NHẪN
  -  5.0 × [37] NGỦ GIẤC NGÀN THU

Tỷ lệ sở thích đối với Tổng điểm tác giả
  - A:J. R. R. Tolkien                        0.500
  - A:Raymond Chandler                        0.500

Tỷ lệ sở thích đối với Tổng điểm thể loại
  - C:Tiểu thuyết                             0.400
  - C:Văn học                                 0.200
  - C:Phiêu lưu                               0.200
  - C:Trinh thám                              0.200

Tỷ lệ sở thích đối với Tổng điểm NXB
  - P:Văn học                                 1.000

Điểm của cuốn sách gần nhất [39] KẺ KHÔNG THỂ GIÃ TỪ:
  cos_A=0.7071  |  cos_C=0.8729  |  cos_P=1.0000
  => AVERAGE = (cos_A + cos_C + cos_P) / 3 = 0.859993
